# Poem text length comparison

Compares poem text length between two parser runs (JSON vs JSONL outputs) across a batch of anthologies. Large differences flag boundary detection issues; empty or very short texts flag missed extractions.

In [ ]:
PAIRS = [
    {
        "name": f"englishpoets{n:02d}wardiala",
        "json":  f"pipeline_output/marker_poems/englishpoets{n:02d}wardiala_poems.json",
        "jsonl": f"pipeline_output/marker_poems/englishpoets{n:02d}wardiala_poems.jsonl",
    }
    for n in range(1, 6)
]

# Poems shorter than this character count are flagged as possibly missed
SHORT_TEXT_THRESHOLD = 50

In [ ]:
import json
from pathlib import Path

import pandas as pd


def _metrics(text: str) -> dict:
    chars = len(text)
    lines = [l for l in text.splitlines() if l.strip()]
    words = len(text.split())
    return {"chars": chars, "words": words, "lines": len(lines)}


def load_json(path, label):
    with open(path, encoding="utf-8") as f:
        poems = json.load(f)
    rows = []
    for p in poems:
        m = _metrics(p.get("text", ""))
        rows.append({
            "title": p.get("title", ""),
            "source_page": p.get("source_page", 0),
            f"chars_{label}": m["chars"],
            f"words_{label}": m["words"],
            f"lines_{label}": m["lines"],
        })
    return pd.DataFrame(rows)


def load_jsonl(path, label):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            p = json.loads(line)
            m = _metrics(p.get("text", ""))
            rows.append({
                "title": p.get("title", ""),
                "source_page": p.get("source_page", 0),
                f"chars_{label}": m["chars"],
                f"words_{label}": m["words"],
                f"lines_{label}": m["lines"],
            })
    return pd.DataFrame(rows)


def process_pair(name, json_path, jsonl_path):
    a = load_json(json_path,   "a")
    b = load_jsonl(jsonl_path, "b")
    merged = a.merge(b, on=["title", "source_page"], how="outer", indicator=True)
    both = merged[merged["_merge"] == "both"].copy()
    both.insert(0, "anthology", name)
    both["char_diff"]  = both["chars_b"]  - both["chars_a"]
    both["char_ratio"] = both.apply(
        lambda r: r["chars_b"] / r["chars_a"] if r["chars_a"] > 0 else float("nan"), axis=1
    )
    return both, merged


all_both = []
all_unmatched = []

for pair in PAIRS:
    both, merged = process_pair(pair["name"], pair["json"], pair["jsonl"])
    all_both.append(both)
    unmatched = merged[merged["_merge"] != "both"].copy()
    unmatched.insert(0, "anthology", pair["name"])
    all_unmatched.append(unmatched)
    print(f"{pair['name']}: {len(both)} matched, {(merged['_merge'] != 'both').sum()} unmatched")

both_df      = pd.concat(all_both,      ignore_index=True)
unmatched_df = pd.concat(all_unmatched, ignore_index=True)

## Per-anthology summary

In [ ]:
summary = (
    both_df.groupby("anthology")
    .agg(
        poems=("title", "count"),
        mean_chars_a=("chars_a", "mean"),
        mean_chars_b=("chars_b", "mean"),
        mean_char_diff=("char_diff", "mean"),
        mean_char_ratio=("char_ratio", "mean"),
        short_a=(  "chars_a", lambda x: (x < SHORT_TEXT_THRESHOLD).sum()),
        short_b=(  "chars_b", lambda x: (x < SHORT_TEXT_THRESHOLD).sum()),
    )
    .reset_index()
)

totals = pd.DataFrame([{
    "anthology": "TOTAL",
    "poems": both_df.shape[0],
    "mean_chars_a":   both_df["chars_a"].mean(),
    "mean_chars_b":   both_df["chars_b"].mean(),
    "mean_char_diff": both_df["char_diff"].mean(),
    "mean_char_ratio":both_df["char_ratio"].mean(),
    "short_a": (both_df["chars_a"] < SHORT_TEXT_THRESHOLD).sum(),
    "short_b": (both_df["chars_b"] < SHORT_TEXT_THRESHOLD).sum(),
}])

display(
    pd.concat([summary, totals], ignore_index=True)
    .style.format({
        "mean_chars_a":    "{:.0f}",
        "mean_chars_b":    "{:.0f}",
        "mean_char_diff":  "{:+.0f}",
        "mean_char_ratio": "{:.3f}",
    })
)

## Largest text length differences (potential boundary issues)

In [ ]:
TOP_N = 20

biggest_diffs = (
    both_df[["anthology", "title", "source_page", "chars_a", "chars_b", "char_diff", "char_ratio"]]
    .assign(abs_diff=both_df["char_diff"].abs())
    .sort_values("abs_diff", ascending=False)
    .drop(columns="abs_diff")
    .head(TOP_N)
    .reset_index(drop=True)
)

display(biggest_diffs.style.format({
    "char_diff":  "{:+d}",
    "char_ratio": "{:.2f}",
}))

## Short or empty texts (possible missed extractions)

In [ ]:
short = both_df[
    (both_df["chars_a"] < SHORT_TEXT_THRESHOLD) |
    (both_df["chars_b"] < SHORT_TEXT_THRESHOLD)
][["anthology", "title", "source_page", "chars_a", "chars_b", "char_diff"]].reset_index(drop=True)

print(f"{len(short)} poem(s) with text shorter than {SHORT_TEXT_THRESHOLD} chars in either version")
display(short)

## Unmatched poems

In [ ]:
print(f"{len(unmatched_df)} unmatched poem(s) across all pairs")
display(unmatched_df[["anthology", "title", "source_page", "_merge"]].reset_index(drop=True))

## Export to CSV (optional)

In [ ]:
OUTPUT_CSV = "text_length_diff.csv"  # set to None to skip

if OUTPUT_CSV:
    both_df[["anthology", "title", "source_page",
             "chars_a", "words_a", "lines_a",
             "chars_b", "words_b", "lines_b",
             "char_diff", "char_ratio"]]\
        .to_csv(OUTPUT_CSV, index=False)
    print(f"Saved to {OUTPUT_CSV}")